# Two-Stage Classification Approach

## Overview
Based on the Grad-CAM findings showing confusion between glioma and 
meningioma, this notebook explores a two-stage classification approach 
recommended by the course professor.

## Approach
- **Stage 1:** Binary classification - tumor vs no tumor
- **Stage 2:** If tumor detected, classify into glioma, meningioma, or pituitary

## Motivation
Separating the "easy" notumor detection from the "hard" tumor type 
classification might allow each stage to specialize and potentially 
improve performance on the harder distinctions.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from google.colab import drive

In [2]:
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_dir = "/content/drive/MyDrive/brain-tumor-data/Training"
test_dir = "/content/drive/MyDrive/brain-tumor-data/Testing"

tumor_classes = ["glioma", "meningioma", "pituitary"]
all_classes = ["glioma", "meningioma", "notumor", "pituitary"]

# Stage 1: binary labels (0 = no tumor, 1 = tumor)
binary_label = {"glioma": 1, "meningioma": 1, "notumor": 0, "pituitary": 1}

# Stage 2: tumor type labels
tumor_label = {"glioma": 0, "meningioma": 1, "pituitary": 2}

# Parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

print("Setup done!")

Mounted at /content/drive
Using device: cuda
Setup done!


In [3]:
class BinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        
        for cls in all_classes:
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(binary_label[cls])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [4]:
class TumorTypeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        
        for cls in tumor_classes:
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(tumor_label[cls])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Stage 1 dataloaders
binary_train = BinaryDataset(train_dir, transform=train_transform)
binary_test = BinaryDataset(test_dir, transform=test_transform)
binary_train_loader = DataLoader(binary_train, batch_size=BATCH_SIZE, shuffle=True)
binary_test_loader = DataLoader(binary_test, batch_size=BATCH_SIZE, shuffle=False)

# Stage 2 dataloaders
tumor_train = TumorTypeDataset(train_dir, transform=train_transform)
tumor_test = TumorTypeDataset(test_dir, transform=test_transform)
tumor_train_loader = DataLoader(tumor_train, batch_size=BATCH_SIZE, shuffle=True)
tumor_test_loader = DataLoader(tumor_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Stage 1 - Training: {len(binary_train)}, Testing: {len(binary_test)}")
print(f"Stage 2 - Training: {len(tumor_train)}, Testing: {len(tumor_test)}")

Stage 1 - Training: 5600, Testing: 1600
Stage 2 - Training: 4200, Testing: 1200


In [6]:
def build_model(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

stage1_model = build_model(2)  # tumor vs no tumor
stage2_model = build_model(3)  # glioma, meningioma, pituitary

print("Stage 1 model:", stage1_model.fc)
print("Stage 2 model:", stage2_model.fc)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s] 


Stage 1 model: Linear(in_features=2048, out_features=2, bias=True)
Stage 2 model: Linear(in_features=2048, out_features=3, bias=True)


In [7]:
def train_model(model, train_loader, epochs, learning_rate, save_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=learning_rate)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        
        acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f} Accuracy: {acc:.2f}%")
        
        # Save after every epoch!
        torch.save(model.state_dict(), save_path)
        print(f"  → Saved to {save_path}")
    
    return model

## Stage 1: Tumor Detection
Training a binary ResNet50 classifier to detect presence of any tumor.

In [9]:
print("=== Training Stage 1: Tumor vs No Tumor ===")
stage1_model = train_model(
    stage1_model, 
    binary_train_loader, 
    EPOCHS, 
    LEARNING_RATE,
    "/content/drive/MyDrive/brain-tumor-data/stage1_model.pth"
)

## Stage 2: Tumor Type Classification
Training a 3-class ResNet50 classifier (glioma, meningioma, pituitary) 
only on images that contain a tumor.

In [11]:
print("=== Training Stage 2: Tumor Type Classification ===")
stage2_model = train_model(
    stage2_model,
    tumor_train_loader,
    EPOCHS,
    LEARNING_RATE,
    "/content/drive/MyDrive/brain-tumor-data/stage2_model.pth"
)

In [8]:
stage1_model = build_model(2)
stage1_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage1_model.pth", map_location=device))
stage1_model = stage1_model.to(device)
stage1_model.eval()
print("Stage 1 loaded!")

stage2_model = build_model(3)
stage2_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage2_model.pth", map_location=device))
stage2_model = stage2_model.to(device)
stage2_model.eval()
print("Stage 2 loaded!")

Stage 1 loaded!
Stage 2 loaded!


## Combined Pipeline Evaluation

**Results:** 85.50% overall accuracy. Compared to baseline (85.75%), the 
two-stage approach improved glioma recall significantly (69% → 75.2%) but 
slightly reduced meningioma recall (80.5% → 74.5%), with similar overall 
accuracy. This suggests the two-stage approach helps for the most 
clinically critical class (glioma) at a small trade-off elsewhere.

In [11]:
def evaluate_twostage(stage1_model, stage2_model, test_dir):
    stage1_model.eval()
    stage2_model.eval()
    
    all_preds = []
    all_labels = []
    
    final_class_to_idx = {"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}
    tumor_idx_to_class = {0: "glioma", 1: "meningioma", 2: "pituitary"}
    
    for cls in all_classes:
        cls_path = os.path.join(test_dir, cls)
        true_label = final_class_to_idx[cls]
        print(f"Processing {cls}...")
        
        for img_file in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_file)
            img = Image.open(img_path).convert("RGB")
            tensor = test_transform(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                # Stage 1
                s1_out = stage1_model(tensor)
                _, s1_pred = torch.max(s1_out, 1)
                
                if s1_pred.item() == 0:
                    final_pred = final_class_to_idx["notumor"]
                else:
                    # Stage 2
                    s2_out = stage2_model(tensor)
                    _, s2_pred = torch.max(s2_out, 1)
                    final_pred = final_class_to_idx[tumor_idx_to_class[s2_pred.item()]]
            
            all_preds.append(final_pred)
            all_labels.append(true_label)
        
        # Show progress per class
        cls_preds = all_preds[-400:]
        cls_correct = sum(p == true_label for p in cls_preds)
        print(f"  {cls}: {cls_correct}/400 correct ({100*cls_correct/400:.1f}%)")
    
    accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f"\nTwo-stage Test Accuracy: {accuracy:.2f}%")
    print(classification_report(all_labels, all_preds, target_names=all_classes))
    
    return all_preds, all_labels

all_preds, all_labels = evaluate_twostage(stage1_model, stage2_model, test_dir)

Processing glioma...
  glioma: 301/400 correct (75.2%)
Processing meningioma...
  meningioma: 298/400 correct (74.5%)
Processing notumor...
  notumor: 389/400 correct (97.2%)
Processing pituitary...
  pituitary: 380/400 correct (95.0%)

Two-stage Test Accuracy: 85.50%
              precision    recall  f1-score   support

      glioma       0.83      0.75      0.79       400
  meningioma       0.78      0.74      0.76       400
     notumor       0.90      0.97      0.94       400
   pituitary       0.90      0.95      0.92       400

    accuracy                           0.85      1600
   macro avg       0.85      0.85      0.85      1600
weighted avg       0.85      0.85      0.85      1600



## Fine-tuning Stage 1

Following the success of fine-tuning the baseline ResNet50, the same 
approach was applied to Stage 1. The last convolutional block (layer4) 
and final classifier were unfrozen and retrained with a lower learning 
rate (0.0001).

**Result:** Training accuracy improved from 96.46% → 99.70%, reducing 
missed tumors from 34 to 13 out of 1200 in the final pipeline.

In [8]:
# Load saved Stage 1 model
finetuned_stage1 = build_model(2)
finetuned_stage1.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage1_model.pth", map_location=device))
finetuned_stage1 = finetuned_stage1.to(device)

# Freeze all layers first
for param in finetuned_stage1.parameters():
    param.requires_grad = False

# Unfreeze layer4 and fc
for param in finetuned_stage1.layer4.parameters():
    param.requires_grad = True
for param in finetuned_stage1.fc.parameters():
    param.requires_grad = True

# Check trainable parameters
trainable = sum(p.numel() for p in finetuned_stage1.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetuned_stage1.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

# Train
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetuned_stage1.parameters()),
    lr=0.0001
)

print("=== Fine-tuning Stage 1 ===")
for epoch in range(EPOCHS):
    finetuned_stage1.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in binary_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = finetuned_stage1(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(binary_train_loader):.4f} Accuracy: {acc:.2f}%")

torch.save(finetuned_stage1.state_dict(), "/content/drive/MyDrive/brain-tumor-data/stage1_finetuned.pth")
print("Fine-tuned Stage 1 saved!")

Trainable parameters: 14,968,834 / 23,512,130
=== Fine-tuning Stage 1 ===
Epoch [1/10] Loss: 0.0823 Accuracy: 97.30%
Epoch [2/10] Loss: 0.0348 Accuracy: 98.82%
Epoch [3/10] Loss: 0.0240 Accuracy: 99.20%
Epoch [4/10] Loss: 0.0204 Accuracy: 99.43%
Epoch [5/10] Loss: 0.0168 Accuracy: 99.43%
Epoch [6/10] Loss: 0.0184 Accuracy: 99.34%
Epoch [7/10] Loss: 0.0170 Accuracy: 99.38%
Epoch [8/10] Loss: 0.0175 Accuracy: 99.48%
Epoch [9/10] Loss: 0.0105 Accuracy: 99.68%
Epoch [10/10] Loss: 0.0082 Accuracy: 99.70%
Fine-tuned Stage 1 saved!


## Fine-tuning Stage 2

After identifying that ResNet50 fine-tuning significantly improved the 
baseline model, the same approach was applied to Stage 2. The last 
convolutional block (layer4) and final classifier were unfrozen and 
retrained with a lower learning rate (0.0001).

**Result:** Training accuracy improved from 85.10% to 98.90%, showing 
the tumor-type classifier benefits substantially from learning 
MRI-specific features rather than relying solely on frozen ImageNet features.

In [9]:
# Load saved Stage 2 model
finetuned_stage2 = build_model(3)
finetuned_stage2.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage2_model.pth", map_location=device))
finetuned_stage2 = finetuned_stage2.to(device)

# Freeze all layers first
for param in finetuned_stage2.parameters():
    param.requires_grad = False

# Unfreeze layer4 and fc
for param in finetuned_stage2.layer4.parameters():
    param.requires_grad = True
for param in finetuned_stage2.fc.parameters():
    param.requires_grad = True

# Check trainable parameters
trainable = sum(p.numel() for p in finetuned_stage2.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetuned_stage2.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

# Train
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetuned_stage2.parameters()),
    lr=0.0001
)

print("=== Fine-tuning Stage 2 ===")
for epoch in range(EPOCHS):
    finetuned_stage2.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tumor_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = finetuned_stage2(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(tumor_train_loader):.4f} Accuracy: {acc:.2f}%")

torch.save(finetuned_stage2.state_dict(), "/content/drive/MyDrive/brain-tumor-data/stage2_finetuned.pth")
print("Fine-tuned Stage 2 saved!")

Trainable parameters: 14,970,883 / 23,514,179
=== Fine-tuning Stage 2 ===
Epoch [1/10] Loss: 0.3053 Accuracy: 88.83%
Epoch [2/10] Loss: 0.1757 Accuracy: 93.12%
Epoch [3/10] Loss: 0.1312 Accuracy: 95.21%
Epoch [4/10] Loss: 0.0944 Accuracy: 96.69%
Epoch [5/10] Loss: 0.0826 Accuracy: 96.74%
Epoch [6/10] Loss: 0.0781 Accuracy: 97.50%
Epoch [7/10] Loss: 0.0658 Accuracy: 97.79%
Epoch [8/10] Loss: 0.0476 Accuracy: 98.33%
Epoch [9/10] Loss: 0.0555 Accuracy: 97.86%
Epoch [10/10] Loss: 0.0387 Accuracy: 98.90%
Fine-tuned Stage 2 saved!
